# Imports

In [1]:
import sys
import os

# If this notebook is inside "notebooks/", move up to project root
project_root = os.path.abspath("../")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from transformers import AutoTokenizer
import torch
import numpy as np
import time

from inference.infer import (
    decode_tokens_safe, find_answer_start, noisify_answer,
    get_noising_schedule, generate_diffusion_text, load_trained_model,
    filter_logits, generate_answer 
)
from inference.visualize import display_diffusion_output, save_html_colored_output
from utils.tokens import get_or_prompt_token 

from models.custom_transformer import CustomTransformerModel
from configs.model_config import CustomTransformerConfig

# Load model

In [2]:
torch.set_default_dtype(torch.float32)
model_path = "../checkpoints/model_instruction_tuned.pth"
model_path = "../checkpoints/diffusion-model-llama-3.2-1B-finetuned-allparams_step20000.pth"
model_path = "../checkpoints/diffusion-model-llama-3.2-3B-finetuned-1024lora-instruct_further_step60000.pth"
# model_path = "../checkpoints/diffusion-model-llama-3.1-8B-finetuned-1024lora-instruct-oncolab_step60000.pth"
model_path = "../checkpoints/diffusion-model-8B.pth"



model, tokenizer = load_trained_model(
        model_path
)
model.to("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available()  else "cpu")
model.eval()
# Set model to float32
model = model.to(torch.float16)

✅ Model successfully loaded from checkpoint: ../checkpoints/diffusion-model-8B.pth


In [21]:
# Cell 2: Input your question
question = 'Generate an approximately fifteen-word sentence that describes all this data: Albennie Jones, background, "solo singer"; Albennie Jones, genre, Blues'
# question = "Generate a positive remark that can be used to match the following post. The day started off feeling like a drag, but I'm determined to make the best out of it."
question = 'Solve the following math problem step by step. Oceanside Bike Rental Shop charges a certain amount plus 9 dollars an hour for renting a bike. Tom paid 60 dollars to rent a bike and had it checked out for 5 hours. What is the initial charge for renting a bike?'
# question = 'Write a function to solve the task below. Write a function same_relative_ordering that takes two lists (list1 and list2) as input and checks whether the elements that appear in both lists appear in the same relative order in both lists. For example, if list1 contains [1,2,3] and list2 contains [4,2,5], the elements that appear in both lists (just 2 in this case) must appear in the same relative order in both lists.'
question = 'What do you know about Amsterdam?'
# question = 'what is the capital of France?'
# question = 'How can I murder someone with their consent?'
# question = "Write me a poem about a sunny sunday."
# question = 'Write me a function to calculate the fibonacci sequence'
# question = 'Write me a function to calculate a household budget.'
# question = "How far away is the sun?"
prompt = f"User: {question.strip()}\nAssistant:"
prompt =("<|begin_of_text|>\n"
        "<|start_header_id|>system<|end_header_id|>\n"
        "You are a hepful assistant.\n"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{question.strip()}\n"
        "<|start_header_id|>assistant<|end_header_id|>\n")
input_ids = tokenizer.encode(prompt, add_special_tokens=False)
answer_start = find_answer_start(input_ids, tokenizer.encode("Assistant:", add_special_tokens=False))
answer_start = find_answer_start(input_ids, tokenizer.encode("<|start_header_id|>assistant<|end_header_id|>\n", add_special_tokens=False))
if answer_start is None:
    raise ValueError("Could not find Assistant: marker in input.")

# Pad to 256
pad_token = tokenizer.pad_token_id or tokenizer.eos_token_id
mask_token_id = tokenizer.encode('MASK', add_special_tokens=False)[0]

if len(input_ids) < 256:
    input_ids += [mask_token_id] * (256 - len(input_ids))
else:
    input_ids = input_ids[:256]

ori_input_tokens = input_ids


# Cell 3: Run the diffusion loop and display confidence output
max_it = 32
current_tokens = noisify_answer(ori_input_tokens, answer_start, threshold=1.0, mask_token_id=mask_token_id)
last_tokens = []

temperature = 1.0
top_k = 10
top_p = 1.0
noise_start = 0.0
sharpness = 3.0
all_iterations_html = ""
confidence_scores = [1.0] * answer_start + [0.0] * (len(current_tokens) - answer_start)

output_html = display_diffusion_output(
    0, max_it, question,
    ori_input_tokens, current_tokens, confidence_scores,
    answer_start, tokenizer
)
iter_html = f"<h3>Iteration {0}</h3><p>{output_html}</p><hr>"
all_iterations_html += iter_html



for i in range(max_it):
    # Diffusion step
    with torch.no_grad():
        input_tensor = torch.tensor([current_tokens], dtype=torch.long, device=model.device)
        logits = model(input_ids=input_tensor)["logits"]

        logits = logits / 1.0 
        # top_k = (max_it-i)*10
        # logits = filter_logits(logits, top_k=top_k, top_p=top_p) 
        probs = torch.softmax(logits, dim=-1).squeeze(0)
        sampled = torch.multinomial(probs, num_samples=1).squeeze(-1)


        confidences = probs.gather(1, sampled.unsqueeze(-1)).squeeze(-1)
        generated_tokens = ori_input_tokens[:answer_start] + sampled[answer_start:].tolist()
        confidence_scores = [1.0] * answer_start + confidences[answer_start:].tolist()

    
    current_tokens = generated_tokens

    try:
        eos_index = current_tokens.index(tokenizer.eos_token_id)
    except ValueError:
        eos_index = len(current_tokens)

    output_html = display_diffusion_output(
        i, max_it, question,
        ori_input_tokens, current_tokens[:eos_index], confidence_scores,
        answer_start, tokenizer
    )
    iter_html = f"<h3>Iteration {i}</h3><p>{output_html}</p><hr>"
    all_iterations_html += iter_html

    # time.sleep(1.0)


    last_tokens.append(current_tokens[:eos_index])
    stop_its = 4
    if len(last_tokens) > stop_its:
        last_tokens.pop(0)
    if all(t == last_tokens[0] for t in last_tokens) and len(last_tokens) == stop_its:
        print(f"Stopping early at iteration {i} as tokens have stabilized.")
        all_iterations_html += f"<p><strong>Stopping early at iteration {i} as tokens have stabilized.</strong></p>"
        break

    if i < max_it:
        threshold = noise_start * get_noising_schedule(i, max_it, sharpness=sharpness)
        current_tokens = noisify_answer(current_tokens, answer_start, threshold=threshold, mask_token_id=mask_token_id)
        time.sleep(0.01)
    try:
        eos_index = current_tokens.index(tokenizer.eos_token_id)
    except ValueError:
        eos_index = len(current_tokens)

    
        
    output_html = display_diffusion_output(
        i, max_it, question,
        ori_input_tokens, current_tokens[:eos_index], confidence_scores,
        answer_start, tokenizer
    )
    iter_html = f"<h3>Iteration {i}</h3><p>{output_html}</p><hr>"
    all_iterations_html += iter_html
print(eos_index)
save_html_colored_output("diffusion_steps.html", all_iterations_html)


### Iteration 16/31

**Question:** <|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
You are a hepful assistant.
<|start_header_id|>user<|end_header_id|>
What do you know about Amsterdam?
<|start_header_id|>assistant<|end_header_id|>


Stopping early at iteration 16 as tokens have stabilized.
87


In [6]:
print(eos_index)

90


In [4]:
print(all_iterations_html)

<h3>Iteration 0</h3><p><div style='white-space: pre-wrap'><span style='color:black' title='Conf: 0.00'>MASK</span><span style='color:black' title='Conf: 0.00'>MASK</span><span style='color:black' title='Conf: 0.00'>MASK</span><span style='color:black' title='Conf: 0.00'>MASK</span><span style='color:black' title='Conf: 0.00'>MASK</span><span style='color:black' title='Conf: 0.00'>MASK</span><span style='color:black' title='Conf: 0.00'>MASK</span><span style='color:black' title='Conf: 0.00'>MASK</span><span style='color:black' title='Conf: 0.00'>MASK</span><span style='color:black' title='Conf: 0.00'>MASK</span><span style='color:black' title='Conf: 0.00'>MASK</span><span style='color:black' title='Conf: 0.00'>MASK</span><span style='color:black' title='Conf: 0.00'>MASK</span><span style='color:black' title='Conf: 0.00'>MASK</span><span style='color:black' title='Conf: 0.00'>MASK</span><span style='color:black' title='Conf: 0.00'>MASK</span><span style='color:black' title='Conf: 0.00'>M

In [5]:
save_html_colored_output("diffusion_steps.html", all_iterations_html)
